# **Ablation Study: GRAFTS and TabFGT Component Analysis**

In [1]:
import os
import json
import time
import copy
import random
import warnings
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    log_loss,
    matthews_corrcoef,
    confusion_matrix,
    balanced_accuracy_score,
    brier_score_loss,
)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings("ignore")


# ============================================================
# ABLATION GLOBAL CONFIG
# ============================================================

# Full-feature files are produced by the train/test split cell.
TRAIN_PATH_ALL = "/content/train.csv"
TEST_PATH_ALL  = "/content/test.csv"

# GRAFTS-selected files are produced by the GRAFTS
TRAIN_PATH_GRAFTS = "/content/train_selected.csv"
TEST_PATH_GRAFTS  = "/content/test_selected.csv"

TARGET_COLUMN = "lung_cancer_risk"
ID_COLUMNS = []
DROP_COLUMNS = []

OUTPUT_DIR = "/content/ablation_outputs"
SAVE_ABLATION_OUTPUTS = True

SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_WORKERS = 0
USE_AMP = True

OUTER_FOLDS = 5
OUTER_DEV_VALID_SIZE = 0.12
FINAL_VALID_SIZE = 0.10
FINAL_ENSEMBLE_SEEDS = [42]

MAX_EPOCHS_ABLATION = 12
PATIENCE_ABLATION = 3

THRESHOLD_GRID = np.linspace(0.16, 0.60, 177)
ACC_DROP_TOL = 0.0015
MIN_DYNAMIC_PRECISION = 0.970
PRECISION_RELAX = 0.012

USE_MILD_POS_WEIGHT = True
POS_WEIGHT_POWER = 0.35
POS_WEIGHT_MAX = 1.25

FIXED_HP = {
    "embed_dim": 192,
    "num_heads": 4,
    "depth": 3,
    "ff_mult": 2.0,
    "dropout": 0.10,
    "token_dropout": 0.02,
    "lr": 8.0e-4,
    "weight_decay": 1e-5,
    "batch_size": 1024,
}

ABLATION_VARIANTS = [
    {
        "variant_id": "A00",
        "variant_name": "All features | No FID | No GATE | No DUAL",
        "use_grafts": False,
        "use_feature_identity": False,
        "use_gated_ffn": False,
        "use_dual_view_readout": False,
    },
    {
        "variant_id": "A01",
        "variant_name": "All features | No FID | No GATE | DUAL",
        "use_grafts": False,
        "use_feature_identity": False,
        "use_gated_ffn": False,
        "use_dual_view_readout": True,
    },
    {
        "variant_id": "A02",
        "variant_name": "All features | No FID | GATE | No DUAL",
        "use_grafts": False,
        "use_feature_identity": False,
        "use_gated_ffn": True,
        "use_dual_view_readout": False,
    },
    {
        "variant_id": "A03",
        "variant_name": "All features | No FID | GATE | DUAL",
        "use_grafts": False,
        "use_feature_identity": False,
        "use_gated_ffn": True,
        "use_dual_view_readout": True,
    },
    {
        "variant_id": "A04",
        "variant_name": "All features | FID | No GATE | No DUAL",
        "use_grafts": False,
        "use_feature_identity": True,
        "use_gated_ffn": False,
        "use_dual_view_readout": False,
    },
    {
        "variant_id": "A05",
        "variant_name": "All features | FID | No GATE | DUAL",
        "use_grafts": False,
        "use_feature_identity": True,
        "use_gated_ffn": False,
        "use_dual_view_readout": True,
    },
    {
        "variant_id": "A06",
        "variant_name": "All features | FID | GATE | No DUAL",
        "use_grafts": False,
        "use_feature_identity": True,
        "use_gated_ffn": True,
        "use_dual_view_readout": False,
    },
    {
        "variant_id": "A07",
        "variant_name": "All features | Full TabFGT",
        "use_grafts": False,
        "use_feature_identity": True,
        "use_gated_ffn": True,
        "use_dual_view_readout": True,
    },
    {
        "variant_id": "A08",
        "variant_name": "GRAFTS | No FID | No GATE | No DUAL",
        "use_grafts": True,
        "use_feature_identity": False,
        "use_gated_ffn": False,
        "use_dual_view_readout": False,
    },
    {
        "variant_id": "A09",
        "variant_name": "GRAFTS | No FID | No GATE | DUAL",
        "use_grafts": True,
        "use_feature_identity": False,
        "use_gated_ffn": False,
        "use_dual_view_readout": True,
    },
    {
        "variant_id": "A10",
        "variant_name": "GRAFTS | No FID | GATE | No DUAL",
        "use_grafts": True,
        "use_feature_identity": False,
        "use_gated_ffn": True,
        "use_dual_view_readout": False,
    },
    {
        "variant_id": "A11",
        "variant_name": "GRAFTS | No FID | GATE | DUAL",
        "use_grafts": True,
        "use_feature_identity": False,
        "use_gated_ffn": True,
        "use_dual_view_readout": True,
    },
    {
        "variant_id": "A12",
        "variant_name": "GRAFTS | FID | No GATE | No DUAL",
        "use_grafts": True,
        "use_feature_identity": True,
        "use_gated_ffn": False,
        "use_dual_view_readout": False,
    },
    {
        "variant_id": "A13",
        "variant_name": "GRAFTS | FID | No GATE | DUAL",
        "use_grafts": True,
        "use_feature_identity": True,
        "use_gated_ffn": False,
        "use_dual_view_readout": True,
    },
    {
        "variant_id": "A14",
        "variant_name": "GRAFTS | FID | GATE | No DUAL",
        "use_grafts": True,
        "use_feature_identity": True,
        "use_gated_ffn": True,
        "use_dual_view_readout": False,
    },
    {
        "variant_id": "A15",
        "variant_name": "Full GRAFTS + TabFGT",
        "use_grafts": True,
        "use_feature_identity": True,
        "use_gated_ffn": True,
        "use_dual_view_readout": True,
    },
]


# ============================================================
# BASIC HELPERS
# ============================================================

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

    if torch.backends.cudnn.is_available():
        torch.backends.cudnn.deterministic = False
        torch.backends.cudnn.benchmark = True

    if torch.cuda.is_available():
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True

    try:
        torch.set_float32_matmul_precision("high")
    except Exception:
        pass


def now():
    return time.strftime("%H:%M:%S")


def ensure_dir(path):
    os.makedirs(path, exist_ok=True)


def safe_read_csv(path):
    if path is None or str(path).strip() == "":
        return None
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Missing file: {path}\n"
            "Run the previous data split and GRAFTS feature-selection cells first."
        )
    return pd.read_csv(path)


def to_numeric_df(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for c in out.columns:
        out[c] = pd.to_numeric(out[c], errors="coerce")
    return out


def clip_probs(prob, eps=1e-7):
    return np.clip(np.asarray(prob, dtype=np.float64), eps, 1 - eps)


def safe_auc(y_true, prob):
    if len(np.unique(y_true)) < 2:
        return np.nan
    return float(roc_auc_score(y_true, prob))


def safe_pr_auc(y_true, prob):
    if len(np.unique(y_true)) < 2:
        return np.nan
    return float(average_precision_score(y_true, prob))


def hp_to_str(hp):
    return (
        f"embed={hp['embed_dim']} | heads={hp['num_heads']} | depth={hp['depth']} | "
        f"ff_mult={hp['ff_mult']} | drop={hp['dropout']} | tok_drop={hp['token_dropout']} | "
        f"lr={hp['lr']} | wd={hp['weight_decay']} | bs={hp['batch_size']}"
    )


def count_trainable_params(model):
    return int(sum(p.numel() for p in model.parameters() if p.requires_grad))


def format_value(x, digits=5):
    if x is None:
        return "NA"
    if isinstance(x, (int, np.integer)):
        return str(int(x))
    if isinstance(x, (float, np.floating)):
        if np.isnan(x):
            return "NA"
        return f"{float(x):.{digits}f}"
    return str(x)


def print_text_table(rows, columns, title=None, digits=5):
    if title:
        print("\n" + "=" * 120)
        print(title)
        print("=" * 120)

    if rows is None or len(rows) == 0:
        print("No rows to display.")
        return

    str_rows = []
    for row in rows:
        str_rows.append([format_value(row.get(col, ""), digits=digits) for col in columns])

    widths = []
    for i, col in enumerate(columns):
        max_cell = max([len(str(col))] + [len(r[i]) for r in str_rows])
        widths.append(max_cell)

    header = " | ".join(str(col).ljust(widths[i]) for i, col in enumerate(columns))
    sep = "-+-".join("-" * widths[i] for i in range(len(columns)))
    print(header)
    print(sep)
    for r in str_rows:
        print(" | ".join(r[i].ljust(widths[i]) for i in range(len(columns))))


def metric_score_binary(y_true, y_pred, metric_name="accuracy"):
    if metric_name == "accuracy":
        return accuracy_score(y_true, y_pred)
    if metric_name == "f1":
        return f1_score(y_true, y_pred, zero_division=0)
    if metric_name == "mcc":
        return matthews_corrcoef(y_true, y_pred)
    if metric_name == "balanced_accuracy":
        return balanced_accuracy_score(y_true, y_pred)
    raise ValueError(f"Unsupported metric: {metric_name}")


def _threshold_table(y_true, prob, grid):
    rows = []
    for thr in grid:
        pred = (prob >= thr).astype(int)
        rows.append({
            "threshold": float(thr),
            "accuracy": accuracy_score(y_true, pred),
            "precision": precision_score(y_true, pred, zero_division=0),
            "recall": recall_score(y_true, pred, zero_division=0),
            "f1": f1_score(y_true, pred, zero_division=0),
            "mcc": matthews_corrcoef(y_true, pred),
            "balanced_accuracy": balanced_accuracy_score(y_true, pred),
        })
    return pd.DataFrame(rows)


def _pick_best_threshold_from_df(df_thr):
    df_thr = df_thr.copy()
    df_thr = df_thr.sort_values(
        by=["accuracy", "f1", "recall", "mcc", "precision"],
        ascending=[False, False, False, False, False]
    ).reset_index(drop=True)
    return float(df_thr.iloc[0]["threshold"])


def tune_threshold_constrained(
    y_true,
    prob,
    grid=THRESHOLD_GRID,
    acc_drop_tol=ACC_DROP_TOL,
    min_dynamic_precision=MIN_DYNAMIC_PRECISION,
    precision_relax=PRECISION_RELAX,
):
    y_true = np.asarray(y_true, dtype=int)
    prob = clip_probs(prob)

    df_thr = _threshold_table(y_true, prob, grid)
    best_acc = float(df_thr["accuracy"].max())

    eligible = df_thr[df_thr["accuracy"] >= best_acc - acc_drop_tol].copy()

    if len(eligible) == 0:
        return _pick_best_threshold_from_df(df_thr)

    dynamic_prec = max(
        float(eligible["precision"].max()) - precision_relax,
        min_dynamic_precision - precision_relax,
    )
    eligible2 = eligible[eligible["precision"] >= dynamic_prec].copy()

    if len(eligible2) == 0:
        eligible2 = eligible.copy()

    eligible2 = eligible2.sort_values(
        by=["f1", "recall", "mcc", "accuracy", "precision"],
        ascending=[False, False, False, False, False]
    ).reset_index(drop=True)

    return float(eligible2.iloc[0]["threshold"])


def compute_metrics(y_true, prob, threshold=0.5):
    y_true = np.asarray(y_true, dtype=int)
    prob = clip_probs(prob)
    pred = (prob >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    return {
        "threshold": float(threshold),
        "accuracy": float(accuracy_score(y_true, pred)),
        "precision": float(precision_score(y_true, pred, zero_division=0)),
        "recall": float(recall_score(y_true, pred, zero_division=0)),
        "specificity": float(specificity),
        "f1": float(f1_score(y_true, pred, zero_division=0)),
        "mcc": float(matthews_corrcoef(y_true, pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, pred)),
        "roc_auc": safe_auc(y_true, prob),
        "pr_auc": safe_pr_auc(y_true, prob),
        "logloss": float(log_loss(y_true, prob, labels=[0, 1])),
        "brier": float(brier_score_loss(y_true, prob)),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }


# ============================================================
# PREPROCESSOR
# ============================================================

class FoldPreprocessor:
    def __init__(self):
        self.imputer = SimpleImputer(strategy="median")
        self.scaler = StandardScaler()

    def fit(self, X_df):
        X_num = to_numeric_df(X_df)
        X_imp = self.imputer.fit_transform(X_num)
        self.scaler.fit(X_imp)
        return self

    def transform(self, X_df):
        X_num = to_numeric_df(X_df)
        X_imp = self.imputer.transform(X_num)
        X_scaled = self.scaler.transform(X_imp)
        X_scaled = np.clip(X_scaled, -8.0, 8.0).astype(np.float32)
        return X_scaled

    def fit_transform(self, X_df):
        self.fit(X_df)
        return self.transform(X_df)


# ============================================================
# ABLATION-AWARE MODEL
# ============================================================

class NumericFeatureTokenizer(nn.Module):
    """
    Each scalar feature becomes a learnable token.
    Original TabFGT: token_j = x_j * W_j + B_j + E_j
    Ablated tokenizer: token_j = x_j * W_j + B_j when feature identity is disabled.
    """
    def __init__(self, n_features, embed_dim, token_dropout=0.01, use_feature_identity=True):
        super().__init__()
        self.n_features = n_features
        self.embed_dim = embed_dim
        self.token_dropout = float(token_dropout)
        self.use_feature_identity = bool(use_feature_identity)

        self.weight = nn.Parameter(torch.randn(n_features, embed_dim) * 0.02)
        self.bias = nn.Parameter(torch.zeros(n_features, embed_dim))

        if self.use_feature_identity:
            self.feature_embed = nn.Parameter(torch.randn(n_features, embed_dim) * 0.02)
        else:
            self.register_parameter("feature_embed", None)

        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x):
        tokens = x.unsqueeze(-1) * self.weight.unsqueeze(0) + self.bias.unsqueeze(0)

        if self.use_feature_identity:
            tokens = tokens + self.feature_embed.unsqueeze(0)

        if self.training and self.token_dropout > 0:
            keep = (torch.rand((x.size(0), x.size(1), 1), device=x.device) > self.token_dropout).float()
            tokens = tokens * keep

        tokens = self.norm(tokens)
        return tokens


class TransformerBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, ff_mult=2.0, dropout=0.10, use_gated_ffn=True):
        super().__init__()
        ff_dim = int(embed_dim * ff_mult)
        self.use_gated_ffn = bool(use_gated_ffn)

        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = nn.MultiheadAttention(
            embed_dim=embed_dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True,
        )
        self.drop1 = nn.Dropout(dropout)

        self.norm2 = nn.LayerNorm(embed_dim)
        self.ff1 = nn.Linear(embed_dim, ff_dim * 2)
        self.ff2 = nn.Linear(ff_dim, embed_dim)
        self.drop2 = nn.Dropout(dropout)

        if self.use_gated_ffn:
            self.gate = nn.Sequential(
                nn.LayerNorm(embed_dim),
                nn.Linear(embed_dim, embed_dim),
                nn.Sigmoid(),
            )
        else:
            self.gate = None

    def forward(self, x):
        y = self.norm1(x)
        attn_out, _ = self.attn(y, y, y, need_weights=False)
        x = x + self.drop1(attn_out)

        y = self.norm2(x)
        a, b = self.ff1(y).chunk(2, dim=-1)
        y = a * F.gelu(b)
        y = self.ff2(self.drop2(y))

        if self.use_gated_ffn:
            gate = self.gate(x)
            x = x + gate * y
        else:
            x = x + y

        return x


class TabFGT(nn.Module):
    """
    Ablation-aware TabFGT.
    Switches:
    - use_feature_identity: adds/removes feature identity embedding E_j.
    - use_gated_ffn: adds/removes the gated FFN update.
    - use_dual_view_readout: uses CLS+mean token readout or CLS-only readout.
    """
    def __init__(
        self,
        input_dim,
        embed_dim=160,
        num_heads=4,
        depth=2,
        ff_mult=2.0,
        dropout=0.10,
        token_dropout=0.01,
        use_feature_identity=True,
        use_gated_ffn=True,
        use_dual_view_readout=True,
    ):
        super().__init__()

        self.use_feature_identity = bool(use_feature_identity)
        self.use_gated_ffn = bool(use_gated_ffn)
        self.use_dual_view_readout = bool(use_dual_view_readout)

        self.tokenizer = NumericFeatureTokenizer(
            n_features=input_dim,
            embed_dim=embed_dim,
            token_dropout=token_dropout,
            use_feature_identity=self.use_feature_identity,
        )

        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.cls_bias = nn.Parameter(torch.zeros(1, 1, embed_dim))

        self.blocks = nn.ModuleList([
            TransformerBlock(
                embed_dim=embed_dim,
                num_heads=num_heads,
                ff_mult=ff_mult,
                dropout=dropout,
                use_gated_ffn=self.use_gated_ffn,
            )
            for _ in range(depth)
        ])

        self.final_norm = nn.LayerNorm(embed_dim)

        head_dim = embed_dim * 2 if self.use_dual_view_readout else embed_dim
        hidden2 = max(embed_dim, 128)

        self.head = nn.Sequential(
            nn.LayerNorm(head_dim),
            nn.Linear(head_dim, hidden2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden2, hidden2 // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden2 // 2, 1),
        )

    def forward(self, x):
        tok = self.tokenizer(x)
        cls = self.cls_token.expand(x.size(0), -1, -1) + self.cls_bias
        tok = torch.cat([cls, tok], dim=1)

        for block in self.blocks:
            tok = block(tok)

        tok = self.final_norm(tok)

        cls_tok = tok[:, 0]

        if self.use_dual_view_readout:
            feat_tok = tok[:, 1:]
            mean_tok = feat_tok.mean(dim=1)
            rep = torch.cat([cls_tok, mean_tok], dim=1)
        else:
            rep = cls_tok

        out = self.head(rep).squeeze(1)
        return out


# ============================================================
# TRAINING AND PREDICTION
# ============================================================

def make_loader(X, y=None, batch_size=1024, shuffle=False):
    X_t = torch.tensor(X, dtype=torch.float32)
    if y is None:
        ds = TensorDataset(X_t)
    else:
        y_t = torch.tensor(y, dtype=torch.float32)
        ds = TensorDataset(X_t, y_t)

    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE.type == "cuda"),
        drop_last=False,
    )


@torch.no_grad()
def predict_proba(model, X, batch_size=1024):
    model.eval()
    loader = make_loader(X, y=None, batch_size=batch_size, shuffle=False)
    probs = []

    for batch in loader:
        xb = batch[0].to(DEVICE, non_blocking=True)
        logits = model(xb)
        p = torch.sigmoid(logits).detach().cpu().numpy()
        probs.append(p)

    return np.concatenate(probs).astype(np.float64)


def train_one_model(
    X_train,
    y_train,
    X_valid,
    y_valid,
    hp,
    ablation_cfg,
    max_epochs=12,
    patience=3,
    seed=42,
):
    set_seed(seed)

    model = TabFGT(
        input_dim=X_train.shape[1],
        embed_dim=hp["embed_dim"],
        num_heads=hp["num_heads"],
        depth=hp["depth"],
        ff_mult=hp["ff_mult"],
        dropout=hp["dropout"],
        token_dropout=hp["token_dropout"],
        use_feature_identity=ablation_cfg["use_feature_identity"],
        use_gated_ffn=ablation_cfg["use_gated_ffn"],
        use_dual_view_readout=ablation_cfg["use_dual_view_readout"],
    ).to(DEVICE)

    train_loader = make_loader(X_train, y_train, batch_size=hp["batch_size"], shuffle=True)
    valid_loader = make_loader(X_valid, y_valid, batch_size=max(1024, hp["batch_size"]), shuffle=False)

    pos_count = max(float(np.sum(np.asarray(y_train) == 1)), 1.0)
    neg_count = max(float(np.sum(np.asarray(y_train) == 0)), 1.0)

    if USE_MILD_POS_WEIGHT:
        pos_weight_value = float(np.clip((neg_count / pos_count) ** POS_WEIGHT_POWER, 1.0, POS_WEIGHT_MAX))
    else:
        pos_weight_value = 1.0

    criterion = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor([pos_weight_value], dtype=torch.float32, device=DEVICE)
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=hp["lr"],
        weight_decay=hp["weight_decay"],
    )

    steps_per_epoch = max(1, len(train_loader))
    total_steps = max(1, max_epochs * steps_per_epoch)

    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=hp["lr"],
        total_steps=total_steps,
        pct_start=0.20,
        div_factor=8.0,
        final_div_factor=80.0,
    )

    use_amp_now = bool(USE_AMP and DEVICE.type == "cuda")
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp_now)

    best_state = copy.deepcopy(model.state_dict())
    best_val_loss = np.inf
    best_epoch = 0
    bad_epochs = 0
    min_delta = 5e-4

    for epoch in range(1, max_epochs + 1):
        model.train()

        for xb, yb in train_loader:
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast(enabled=use_amp_now):
                logits = model(xb)
                loss = criterion(logits, yb)

            scaler.scale(loss).backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

        model.eval()
        val_probs = []
        for xb, _ in valid_loader:
            xb = xb.to(DEVICE, non_blocking=True)
            with torch.cuda.amp.autocast(enabled=use_amp_now):
                logits = model(xb)
                p = torch.sigmoid(logits)
            val_probs.append(p.detach().cpu().numpy())

        val_probs = np.concatenate(val_probs).astype(np.float64)
        val_loss = log_loss(y_valid, clip_probs(val_probs), labels=[0, 1])

        if val_loss < best_val_loss - min_delta:
            best_val_loss = float(val_loss)
            best_epoch = int(epoch)
            best_state = copy.deepcopy(model.state_dict())
            bad_epochs = 0
        else:
            bad_epochs += 1

        if bad_epochs >= patience:
            break

    model.load_state_dict(best_state)
    return model, best_epoch, best_val_loss, pos_weight_value, count_trainable_params(model)


# ============================================================
# DATA LOADING PER VARIANT
# ============================================================

def get_variant_paths(cfg):
    if cfg["use_grafts"]:
        return TRAIN_PATH_GRAFTS, TEST_PATH_GRAFTS
    return TRAIN_PATH_ALL, TEST_PATH_ALL


def load_variant_data(cfg):
    train_path, test_path = get_variant_paths(cfg)
    train_df = safe_read_csv(train_path)
    test_df = safe_read_csv(test_path)

    if TARGET_COLUMN not in train_df.columns:
        raise ValueError(f"Target column '{TARGET_COLUMN}' not found in {train_path}.")

    drop_train = list(set(ID_COLUMNS + DROP_COLUMNS + [TARGET_COLUMN]))
    X_train_df = train_df.drop(columns=[c for c in drop_train if c in train_df.columns], errors="ignore").copy()
    y_raw = train_df[TARGET_COLUMN].copy()

    le = LabelEncoder()
    y = le.fit_transform(y_raw)

    if len(le.classes_) != 2:
        raise ValueError(f"Binary classification only. Found classes: {list(le.classes_)}")

    class_names = [str(x) for x in le.classes_]

    drop_test = list(set(ID_COLUMNS + DROP_COLUMNS + [TARGET_COLUMN]))
    X_test_df = test_df.drop(columns=[c for c in drop_test if c in test_df.columns], errors="ignore").copy()

    if TARGET_COLUMN in test_df.columns:
        y_test = le.transform(test_df[TARGET_COLUMN])
        has_test_labels = True
    else:
        y_test = None
        has_test_labels = False

    return {
        "train_path": train_path,
        "test_path": test_path,
        "train_df": train_df,
        "test_df": test_df,
        "X_train_df": X_train_df,
        "y": y,
        "X_test_df": X_test_df,
        "y_test": y_test,
        "has_test_labels": has_test_labels,
        "class_names": class_names,
    }


# ============================================================
# ONE VARIANT ABLATION RUN
# ============================================================

def run_one_ablation_variant(cfg, hp=FIXED_HP):
    data = load_variant_data(cfg)
    X_train_df = data["X_train_df"]
    y = data["y"]
    X_test_df = data["X_test_df"]
    y_test = data["y_test"]
    has_test_labels = data["has_test_labels"]

    print("\n" + "=" * 120)
    print(f"[{now()}] RUNNING {cfg['variant_id']} | {cfg['variant_name']}")
    print("=" * 120)
    print(f"GRAFTS selected features : {cfg['use_grafts']}")
    print(f"Feature identity         : {cfg['use_feature_identity']}")
    print(f"Gated FFN                : {cfg['use_gated_ffn']}")
    print(f"Dual-view readout        : {cfg['use_dual_view_readout']}")
    print(f"Train path               : {data['train_path']}")
    print(f"Test path                : {data['test_path']}")
    print(f"Train rows/features      : {X_train_df.shape[0]} / {X_train_df.shape[1]}")
    print(f"Device                   : {DEVICE}")
    print(f"Fixed HP                 : {hp_to_str(hp)}")

    set_seed(SEED)

    outer_cv = StratifiedKFold(n_splits=OUTER_FOLDS, shuffle=True, random_state=SEED)
    oof_prob = np.zeros(len(y), dtype=np.float64)
    fold_rows = []
    param_count_last = None

    for fold_id, (dev_idx, hold_idx) in enumerate(outer_cv.split(X_train_df, y), start=1):
        X_dev_df = X_train_df.iloc[dev_idx].reset_index(drop=True)
        y_dev = y[dev_idx]
        X_hold_df = X_train_df.iloc[hold_idx].reset_index(drop=True)
        y_hold = y[hold_idx]

        X_tr_df, X_va_df, y_tr, y_va = train_test_split(
            X_dev_df,
            y_dev,
            test_size=OUTER_DEV_VALID_SIZE,
            random_state=SEED + fold_id,
            stratify=y_dev,
        )

        prep = FoldPreprocessor()
        X_tr = prep.fit_transform(X_tr_df)
        X_va = prep.transform(X_va_df)
        X_hold = prep.transform(X_hold_df)

        model, best_epoch, best_val_loss, pos_weight_value, param_count = train_one_model(
            X_tr,
            y_tr,
            X_va,
            y_va,
            hp=hp,
            ablation_cfg=cfg,
            max_epochs=MAX_EPOCHS_ABLATION,
            patience=PATIENCE_ABLATION,
            seed=SEED + fold_id * 1000,
        )
        param_count_last = param_count

        va_prob = predict_proba(model, X_va, batch_size=max(1024, hp["batch_size"]))
        fold_threshold = tune_threshold_constrained(
            y_true=y_va,
            prob=va_prob,
            grid=THRESHOLD_GRID,
            acc_drop_tol=ACC_DROP_TOL,
            min_dynamic_precision=MIN_DYNAMIC_PRECISION,
            precision_relax=PRECISION_RELAX,
        )

        hold_prob = predict_proba(model, X_hold, batch_size=max(1024, hp["batch_size"]))
        oof_prob[hold_idx] = hold_prob

        fold_metrics = compute_metrics(y_hold, hold_prob, threshold=fold_threshold)
        fold_metrics.update({
            "variant_id": cfg["variant_id"],
            "fold": fold_id,
            "best_epoch": best_epoch,
            "best_val_loss": best_val_loss,
            "pos_weight": float(pos_weight_value),
            "n_features": int(X_train_df.shape[1]),
            "params": int(param_count),
        })
        fold_rows.append(fold_metrics)

        print(
            f"  fold {fold_id}/{OUTER_FOLDS} | "
            f"acc={fold_metrics['accuracy']:.5f} | "
            f"prec={fold_metrics['precision']:.5f} | "
            f"rec={fold_metrics['recall']:.5f} | "
            f"f1={fold_metrics['f1']:.5f} | "
            f"auc={fold_metrics['roc_auc']:.5f} | "
            f"logloss={fold_metrics['logloss']:.5f} | "
            f"thr={fold_threshold:.4f}"
        )

    oof_threshold = tune_threshold_constrained(
        y_true=y,
        prob=oof_prob,
        grid=THRESHOLD_GRID,
        acc_drop_tol=ACC_DROP_TOL,
        min_dynamic_precision=MIN_DYNAMIC_PRECISION,
        precision_relax=PRECISION_RELAX,
    )
    oof_metrics = compute_metrics(y, oof_prob, threshold=oof_threshold)

    fold_df = pd.DataFrame(fold_rows)
    fold_mean = fold_df[[
        "accuracy", "precision", "recall", "specificity", "f1",
        "mcc", "balanced_accuracy", "roc_auc", "pr_auc", "logloss", "brier"
    ]].mean().to_dict()
    fold_std = fold_df[[
        "accuracy", "precision", "recall", "specificity", "f1",
        "mcc", "balanced_accuracy", "roc_auc", "pr_auc", "logloss", "brier"
    ]].std(ddof=1).to_dict()

    print(
        f"  OOF summary | acc={oof_metrics['accuracy']:.5f} | "
        f"prec={oof_metrics['precision']:.5f} | rec={oof_metrics['recall']:.5f} | "
        f"f1={oof_metrics['f1']:.5f} | auc={oof_metrics['roc_auc']:.5f} | "
        f"logloss={oof_metrics['logloss']:.5f} | thr={oof_threshold:.4f}"
    )

    # Final training for test-set evaluation.
    final_models = []
    final_preps = []
    final_valid_probs = []
    final_valid_targets = []
    final_param_count = param_count_last

    for i, seed_i in enumerate(FINAL_ENSEMBLE_SEEDS, start=1):
        X_tr_df, X_va_df, y_tr, y_va = train_test_split(
            X_train_df,
            y,
            test_size=FINAL_VALID_SIZE,
            random_state=seed_i,
            stratify=y,
        )

        prep = FoldPreprocessor()
        X_tr = prep.fit_transform(X_tr_df)
        X_va = prep.transform(X_va_df)

        model, best_epoch, best_val_loss, pos_weight_value, param_count = train_one_model(
            X_tr,
            y_tr,
            X_va,
            y_va,
            hp=hp,
            ablation_cfg=cfg,
            max_epochs=MAX_EPOCHS_ABLATION,
            patience=PATIENCE_ABLATION,
            seed=seed_i,
        )
        final_param_count = param_count
        final_models.append(model)
        final_preps.append(prep)

        va_prob = predict_proba(model, X_va, batch_size=max(1024, hp["batch_size"]))
        final_valid_probs.append(va_prob)
        final_valid_targets.append(np.asarray(y_va, dtype=int))

        print(
            f"  final model {i}/{len(FINAL_ENSEMBLE_SEEDS)} | "
            f"best_epoch={best_epoch} | best_val_loss={best_val_loss:.5f} | "
            f"pos_weight={pos_weight_value:.3f}"
        )

    final_valid_prob = np.concatenate(final_valid_probs).astype(np.float64)
    final_valid_y = np.concatenate(final_valid_targets).astype(int)
    final_threshold = tune_threshold_constrained(
        y_true=final_valid_y,
        prob=final_valid_prob,
        grid=THRESHOLD_GRID,
        acc_drop_tol=ACC_DROP_TOL,
        min_dynamic_precision=MIN_DYNAMIC_PRECISION,
        precision_relax=PRECISION_RELAX,
    )

    test_metrics = None
    final_test_prob = None
    final_test_pred = None

    if X_test_df is not None:
        test_probs = []
        for model, prep in zip(final_models, final_preps):
            X_test_proc = prep.transform(X_test_df)
            p = predict_proba(model, X_test_proc, batch_size=max(1024, hp["batch_size"]))
            test_probs.append(p)
        final_test_prob = np.mean(np.vstack(test_probs), axis=0)
        final_test_pred = (final_test_prob >= final_threshold).astype(int)

        if has_test_labels:
            test_metrics = compute_metrics(y_test, final_test_prob, threshold=final_threshold)
            print(
                f"  TEST summary | acc={test_metrics['accuracy']:.5f} | "
                f"prec={test_metrics['precision']:.5f} | rec={test_metrics['recall']:.5f} | "
                f"spec={test_metrics['specificity']:.5f} | f1={test_metrics['f1']:.5f} | "
                f"auc={test_metrics['roc_auc']:.5f} | logloss={test_metrics['logloss']:.5f} | "
                f"thr={final_threshold:.4f}"
            )

    result_row = {
        "variant_id": cfg["variant_id"],
        "variant_name": cfg["variant_name"],
        "grafts": "Yes" if cfg["use_grafts"] else "No",
        "feature_identity": "Yes" if cfg["use_feature_identity"] else "No",
        "gated_ffn": "Yes" if cfg["use_gated_ffn"] else "No",
        "dual_view": "Yes" if cfg["use_dual_view_readout"] else "No",
        "n_features": int(X_train_df.shape[1]),
        "params": int(final_param_count),
        "oof_threshold": float(oof_threshold),
        "test_threshold": float(final_threshold),
    }

    for k, v in oof_metrics.items():
        result_row[f"oof_{k}"] = v
    for k, v in fold_mean.items():
        result_row[f"fold_mean_{k}"] = float(v)
    for k, v in fold_std.items():
        result_row[f"fold_std_{k}"] = float(v)

    if test_metrics is not None:
        for k, v in test_metrics.items():
            result_row[f"test_{k}"] = v

    return {
        "result_row": result_row,
        "fold_df": fold_df,
        "oof_prob": oof_prob,
        "test_prob": final_test_prob,
        "test_pred": final_test_pred,
        "y": y,
        "y_test": y_test,
    }


# ============================================================
# RUN ALL ABLATION VARIANTS
# ============================================================

set_seed(SEED)
ensure_dir(OUTPUT_DIR)

print("\n" + "=" * 120)
print(f"[{now()}] ABLATION STUDY STARTED")
print("=" * 120)
print(f"Device  : {DEVICE}")
print(f"Fixed HP: {hp_to_str(FIXED_HP)}")
print(f"Variants: {len(ABLATION_VARIANTS)}")
print("Note    : This cell prints text-based tables only; CSV copies are also saved for manuscript use.")

all_results = []
all_fold_dfs = []
raw_predictions = {}

for cfg in ABLATION_VARIANTS:
    out = run_one_ablation_variant(cfg, hp=FIXED_HP)
    all_results.append(out["result_row"])
    all_fold_dfs.append(out["fold_df"])
    raw_predictions[cfg["variant_id"]] = {
        "oof_prob": out["oof_prob"],
        "test_prob": out["test_prob"],
        "test_pred": out["test_pred"],
    }

ablation_results_df = pd.DataFrame(all_results)
ablation_fold_results_df = pd.concat(all_fold_dfs, ignore_index=True)

if "test_accuracy" in ablation_results_df.columns:
    ablation_results_df = ablation_results_df.sort_values(
        by=["test_accuracy", "test_f1", "test_roc_auc", "test_logloss"],
        ascending=[False, False, False, True]
    ).reset_index(drop=True)
else:
    ablation_results_df = ablation_results_df.sort_values(
        by=["oof_accuracy", "oof_f1", "oof_roc_auc", "oof_logloss"],
        ascending=[False, False, False, True]
    ).reset_index(drop=True)

# ============================================================
# TEXTUAL OUTPUT TABLES
# ============================================================

variant_cols = [
    "variant_id", "variant_name", "grafts", "feature_identity",
    "gated_ffn", "dual_view", "n_features", "params"
]
print_text_table(
    ablation_results_df.to_dict("records"),
    variant_cols,
    title="ABLATION DESIGN SUMMARY",
    digits=5,
)

oof_cols = [
    "variant_id", "n_features", "oof_threshold", "oof_accuracy", "oof_precision",
    "oof_recall", "oof_specificity", "oof_f1", "oof_mcc", "oof_balanced_accuracy",
    "oof_roc_auc", "oof_pr_auc", "oof_logloss", "oof_brier"
]
print_text_table(
    ablation_results_df.to_dict("records"),
    oof_cols,
    title="ABLATION OOF METRICS TEXT TABLE",
    digits=5,
)

if "test_accuracy" in ablation_results_df.columns:
    test_cols = [
        "variant_id", "n_features", "test_threshold", "test_accuracy", "test_precision",
        "test_recall", "test_specificity", "test_f1", "test_mcc", "test_balanced_accuracy",
        "test_roc_auc", "test_pr_auc", "test_logloss", "test_brier", "test_tn", "test_fp", "test_fn", "test_tp"
    ]
    print_text_table(
        ablation_results_df.to_dict("records"),
        test_cols,
        title="ABLATION TEST METRICS TEXT TABLE",
        digits=5,
    )

fold_cols = [
    "variant_id", "fold", "threshold", "accuracy", "precision", "recall",
    "specificity", "f1", "mcc", "roc_auc", "logloss", "brier"
]
print_text_table(
    ablation_fold_results_df[fold_cols].to_dict("records"),
    fold_cols,
    title="FOLD-WISE ABLATION RESULTS TEXT TABLE",
    digits=5,
)

full_candidates = ablation_results_df[ablation_results_df["variant_id"] == "A15"]
if len(full_candidates) == 1:
    full_row = full_candidates.iloc[0].to_dict()
    delta_rows = []
    for _, row in ablation_results_df.iterrows():
        row = row.to_dict()
        d = {
            "variant_id": row["variant_id"],
            "variant_name": row["variant_name"],
        }
        for metric in ["accuracy", "precision", "recall", "specificity", "f1", "mcc", "roc_auc", "logloss", "brier"]:
            key = f"test_{metric}"
            if key in row and key in full_row:
                d[f"delta_test_{metric}"] = float(row[key]) - float(full_row[key])
        delta_rows.append(d)

    delta_cols = [
        "variant_id", "delta_test_accuracy", "delta_test_precision", "delta_test_recall",
        "delta_test_specificity", "delta_test_f1", "delta_test_mcc",
        "delta_test_roc_auc", "delta_test_logloss", "delta_test_brier"
    ]
    print_text_table(
        delta_rows,
        delta_cols,
        title="DELTA AGAINST FULL MODEL A15: GRAFTS + FEATURE IDENTITY + GATED FFN + DUAL-VIEW READOUT",
        digits=5,
    )

if SAVE_ABLATION_OUTPUTS:
    ablation_results_path = os.path.join(OUTPUT_DIR, "ablation_summary_metrics.csv")
    fold_results_path = os.path.join(OUTPUT_DIR, "ablation_fold_metrics.csv")
    config_path = os.path.join(OUTPUT_DIR, "ablation_config.json")

    ablation_results_df.to_csv(ablation_results_path, index=False)
    ablation_fold_results_df.to_csv(fold_results_path, index=False)

    with open(config_path, "w") as f:
        json.dump(
            {
                "TARGET_COLUMN": TARGET_COLUMN,
                "TRAIN_PATH_ALL": TRAIN_PATH_ALL,
                "TEST_PATH_ALL": TEST_PATH_ALL,
                "TRAIN_PATH_GRAFTS": TRAIN_PATH_GRAFTS,
                "TEST_PATH_GRAFTS": TEST_PATH_GRAFTS,
                "SEED": SEED,
                "OUTER_FOLDS": OUTER_FOLDS,
                "OUTER_DEV_VALID_SIZE": OUTER_DEV_VALID_SIZE,
                "FINAL_VALID_SIZE": FINAL_VALID_SIZE,
                "MAX_EPOCHS_ABLATION": MAX_EPOCHS_ABLATION,
                "PATIENCE_ABLATION": PATIENCE_ABLATION,
                "FIXED_HP": FIXED_HP,
                "ABLATION_VARIANTS": ABLATION_VARIANTS,
            },
            f,
            indent=2,
        )

    print("\n" + "=" * 120)
    print("SAVED ABLATION OUTPUTS")
    print("=" * 120)
    print("Summary CSV:", ablation_results_path)
    print("Fold CSV   :", fold_results_path)
    print("Config JSON:", config_path)

print("\n" + "=" * 120)
print(f"[{now()}] ABLATION STUDY FINISHED")
print("=" * 120)


[14:52:48] ABLATION STUDY STARTED
Device  : cuda
Fixed HP: embed=192 | heads=4 | depth=3 | ff_mult=2.0 | drop=0.1 | tok_drop=0.02 | lr=0.0008 | wd=1e-05 | bs=1024
Variants: 16
Note    : This cell prints text-based tables only; CSV copies are also saved for manuscript use.

[14:52:48] RUNNING A00 | All features | No FID | No GATE | No DUAL
GRAFTS selected features : False
Feature identity         : False
Gated FFN                : False
Dual-view readout        : False
Train path               : /content/train.csv
Test path                : /content/test.csv
Train rows/features      : 4000 / 29
Device                   : cuda
Fixed HP                 : embed=192 | heads=4 | depth=3 | ff_mult=2.0 | drop=0.1 | tok_drop=0.02 | lr=0.0008 | wd=1e-05 | bs=1024
  fold 1/5 | acc=0.96500 | prec=0.90094 | rec=0.96465 | f1=0.93171 | auc=0.99565 | logloss=0.08852 | thr=0.3850
  fold 2/5 | acc=0.97750 | prec=0.92056 | rec=0.99495 | f1=0.95631 | auc=0.99819 | logloss=0.07081 | thr=0.4325
  fold 3/5 

In [5]:
import os
import json
import time
import copy
import random
import warnings
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    log_loss,
    matthews_corrcoef,
    confusion_matrix,
    balanced_accuracy_score,
    brier_score_loss,
)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings("ignore")


# ============================================================
# ABLATION GLOBAL CONFIG
# ============================================================

TRAIN_PATH_ALL = "/content/train.csv"
TEST_PATH_ALL  = "/content/test.csv"

TRAIN_PATH_GRAFTS = "/content/train_selected.csv"
TEST_PATH_GRAFTS  = "/content/test_selected.csv"

TARGET_COLUMN = "lung_cancer_risk"
ID_COLUMNS = []
DROP_COLUMNS = []

OUTPUT_DIR = "/content/ablation_outputs"
SAVE_ABLATION_OUTPUTS = True

SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_WORKERS = 0
USE_AMP = True

FINAL_VALID_SIZE = 0.10
FINAL_ENSEMBLE_SEEDS = [42]

MAX_EPOCHS_ABLATION = 12
PATIENCE_ABLATION = 3

THRESHOLD_GRID = np.linspace(0.16, 0.60, 177)
ACC_DROP_TOL = 0.0015
MIN_DYNAMIC_PRECISION = 0.970
PRECISION_RELAX = 0.012

USE_MILD_POS_WEIGHT = True
POS_WEIGHT_POWER = 0.35
POS_WEIGHT_MAX = 1.25

FIXED_HP = {
    "embed_dim": 192,
    "num_heads": 4,
    "depth": 3,
    "ff_mult": 2.0,
    "dropout": 0.10,
    "token_dropout": 0.02,
    "lr": 8.0e-4,
    "weight_decay": 1e-5,
    "batch_size": 1024,
}

ABLATION_VARIANTS = [
    {
        "variant_id": "A01",
        "variant_name": "All features | No FID | No GATE | No DUAL",
        "use_grafts": False,
        "use_feature_identity": False,
        "use_gated_ffn": False,
        "use_dual_view_readout": False,
    },
    {
        "variant_id": "A02",
        "variant_name": "All features | No FID | No GATE | DUAL",
        "use_grafts": False,
        "use_feature_identity": False,
        "use_gated_ffn": False,
        "use_dual_view_readout": True,
    },
    {
        "variant_id": "A03",
        "variant_name": "All features | No FID | GATE | DUAL",
        "use_grafts": False,
        "use_feature_identity": False,
        "use_gated_ffn": True,
        "use_dual_view_readout": True,
    },
    {
        "variant_id": "A04",
        "variant_name": "GRAFTS | No FID | No GATE | No DUAL",
        "use_grafts": True,
        "use_feature_identity": False,
        "use_gated_ffn": False,
        "use_dual_view_readout": False,
    },
    {
        "variant_id": "A05",
        "variant_name": "GRAFTS | No FID | No GATE | DUAL",
        "use_grafts": True,
        "use_feature_identity": False,
        "use_gated_ffn": False,
        "use_dual_view_readout": True,
    },
    {
        "variant_id": "A06",
        "variant_name": "GRAFTS | No FID | GATE | No DUAL",
        "use_grafts": True,
        "use_feature_identity": False,
        "use_gated_ffn": True,
        "use_dual_view_readout": False,
    },
    {
        "variant_id": "A07",
        "variant_name": "GRAFTS | No FID | GATE | DUAL",
        "use_grafts": True,
        "use_feature_identity": False,
        "use_gated_ffn": True,
        "use_dual_view_readout": True,
    },
    {
        "variant_id": "A08",
        "variant_name": "Proposed Full GRAFTS + TabFGT",
        "use_grafts": True,
        "use_feature_identity": True,
        "use_gated_ffn": True,
        "use_dual_view_readout": True,
    },
]


# ============================================================
# BASIC HELPERS
# ============================================================

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

    if torch.backends.cudnn.is_available():
        torch.backends.cudnn.deterministic = False
        torch.backends.cudnn.benchmark = True

    if torch.cuda.is_available():
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True

    try:
        torch.set_float32_matmul_precision("high")
    except Exception:
        pass


def now():
    return time.strftime("%H:%M:%S")


def ensure_dir(path):
    os.makedirs(path, exist_ok=True)


def safe_read_csv(path):
    if path is None or str(path).strip() == "":
        return None
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Missing file: {path}\n"
            "Run the previous data split and GRAFTS feature-selection cells first."
        )
    return pd.read_csv(path)


def to_numeric_df(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for c in out.columns:
        out[c] = pd.to_numeric(out[c], errors="coerce")
    return out


def clip_probs(prob, eps=1e-7):
    return np.clip(np.asarray(prob, dtype=np.float64), eps, 1 - eps)


def safe_auc(y_true, prob):
    if len(np.unique(y_true)) < 2:
        return np.nan
    return float(roc_auc_score(y_true, prob))


def safe_pr_auc(y_true, prob):
    if len(np.unique(y_true)) < 2:
        return np.nan
    return float(average_precision_score(y_true, prob))


def hp_to_str(hp):
    return (
        f"embed={hp['embed_dim']} | heads={hp['num_heads']} | depth={hp['depth']} | "
        f"ff_mult={hp['ff_mult']} | drop={hp['dropout']} | tok_drop={hp['token_dropout']} | "
        f"lr={hp['lr']} | wd={hp['weight_decay']} | bs={hp['batch_size']}"
    )


def count_trainable_params(model):
    return int(sum(p.numel() for p in model.parameters() if p.requires_grad))


def format_value(x, digits=5):
    if x is None:
        return "NA"
    if isinstance(x, (int, np.integer)):
        return str(int(x))
    if isinstance(x, (float, np.floating)):
        if np.isnan(x):
            return "NA"
        return f"{float(x):.{digits}f}"
    return str(x)


def print_text_table(rows, columns, title=None, digits=5):
    if title:
        print("\n" + "=" * 120)
        print(title)
        print("=" * 120)

    if rows is None or len(rows) == 0:
        print("No rows to display.")
        return

    str_rows = []
    for row in rows:
        str_rows.append([format_value(row.get(col, ""), digits=digits) for col in columns])

    widths = []
    for i, col in enumerate(columns):
        max_cell = max([len(str(col))] + [len(r[i]) for r in str_rows])
        widths.append(max_cell)

    header = " | ".join(str(col).ljust(widths[i]) for i, col in enumerate(columns))
    sep = "-+-".join("-" * widths[i] for i in range(len(columns)))
    print(header)
    print(sep)
    for r in str_rows:
        print(" | ".join(r[i].ljust(widths[i]) for i in range(len(columns))))


def _threshold_table(y_true, prob, grid):
    rows = []
    for thr in grid:
        pred = (prob >= thr).astype(int)
        rows.append({
            "threshold": float(thr),
            "accuracy": accuracy_score(y_true, pred),
            "precision": precision_score(y_true, pred, zero_division=0),
            "recall": recall_score(y_true, pred, zero_division=0),
            "f1": f1_score(y_true, pred, zero_division=0),
            "mcc": matthews_corrcoef(y_true, pred),
            "balanced_accuracy": balanced_accuracy_score(y_true, pred),
        })
    return pd.DataFrame(rows)


def _pick_best_threshold_from_df(df_thr):
    df_thr = df_thr.copy()
    df_thr = df_thr.sort_values(
        by=["accuracy", "f1", "recall", "mcc", "precision"],
        ascending=[False, False, False, False, False]
    ).reset_index(drop=True)
    return float(df_thr.iloc[0]["threshold"])


def tune_threshold_constrained(
    y_true,
    prob,
    grid=THRESHOLD_GRID,
    acc_drop_tol=ACC_DROP_TOL,
    min_dynamic_precision=MIN_DYNAMIC_PRECISION,
    precision_relax=PRECISION_RELAX,
):
    y_true = np.asarray(y_true, dtype=int)
    prob = clip_probs(prob)

    df_thr = _threshold_table(y_true, prob, grid)
    best_acc = float(df_thr["accuracy"].max())

    eligible = df_thr[df_thr["accuracy"] >= best_acc - acc_drop_tol].copy()

    if len(eligible) == 0:
        return _pick_best_threshold_from_df(df_thr)

    dynamic_prec = max(
        float(eligible["precision"].max()) - precision_relax,
        min_dynamic_precision - precision_relax,
    )
    eligible2 = eligible[eligible["precision"] >= dynamic_prec].copy()

    if len(eligible2) == 0:
        eligible2 = eligible.copy()

    eligible2 = eligible2.sort_values(
        by=["f1", "recall", "mcc", "accuracy", "precision"],
        ascending=[False, False, False, False, False]
    ).reset_index(drop=True)

    return float(eligible2.iloc[0]["threshold"])


def compute_metrics(y_true, prob, threshold=0.5):
    y_true = np.asarray(y_true, dtype=int)
    prob = clip_probs(prob)
    pred = (prob >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    return {
        "threshold": float(threshold),
        "accuracy": float(accuracy_score(y_true, pred)),
        "precision": float(precision_score(y_true, pred, zero_division=0)),
        "recall": float(recall_score(y_true, pred, zero_division=0)),
        "specificity": float(specificity),
        "f1": float(f1_score(y_true, pred, zero_division=0)),
        "mcc": float(matthews_corrcoef(y_true, pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, pred)),
        "roc_auc": safe_auc(y_true, prob),
        "pr_auc": safe_pr_auc(y_true, prob),
        "logloss": float(log_loss(y_true, prob, labels=[0, 1])),
        "brier": float(brier_score_loss(y_true, prob)),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }


# ============================================================
# PREPROCESSOR
# ============================================================

class FoldPreprocessor:
    def __init__(self):
        self.imputer = SimpleImputer(strategy="median")
        self.scaler = StandardScaler()

    def fit(self, X_df):
        X_num = to_numeric_df(X_df)
        X_imp = self.imputer.fit_transform(X_num)
        self.scaler.fit(X_imp)
        return self

    def transform(self, X_df):
        X_num = to_numeric_df(X_df)
        X_imp = self.imputer.transform(X_num)
        X_scaled = self.scaler.transform(X_imp)
        X_scaled = np.clip(X_scaled, -8.0, 8.0).astype(np.float32)
        return X_scaled

    def fit_transform(self, X_df):
        self.fit(X_df)
        return self.transform(X_df)


# ============================================================
# ABLATION-AWARE MODEL
# ============================================================

class NumericFeatureTokenizer(nn.Module):
    def __init__(self, n_features, embed_dim, token_dropout=0.01, use_feature_identity=True):
        super().__init__()
        self.n_features = n_features
        self.embed_dim = embed_dim
        self.token_dropout = float(token_dropout)
        self.use_feature_identity = bool(use_feature_identity)

        self.weight = nn.Parameter(torch.randn(n_features, embed_dim) * 0.02)
        self.bias = nn.Parameter(torch.zeros(n_features, embed_dim))

        if self.use_feature_identity:
            self.feature_embed = nn.Parameter(torch.randn(n_features, embed_dim) * 0.02)
        else:
            self.register_parameter("feature_embed", None)

        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x):
        tokens = x.unsqueeze(-1) * self.weight.unsqueeze(0) + self.bias.unsqueeze(0)

        if self.use_feature_identity:
            tokens = tokens + self.feature_embed.unsqueeze(0)

        if self.training and self.token_dropout > 0:
            keep = (torch.rand((x.size(0), x.size(1), 1), device=x.device) > self.token_dropout).float()
            tokens = tokens * keep

        tokens = self.norm(tokens)
        return tokens


class TransformerBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, ff_mult=2.0, dropout=0.10, use_gated_ffn=True):
        super().__init__()
        ff_dim = int(embed_dim * ff_mult)
        self.use_gated_ffn = bool(use_gated_ffn)

        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = nn.MultiheadAttention(
            embed_dim=embed_dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True,
        )
        self.drop1 = nn.Dropout(dropout)

        self.norm2 = nn.LayerNorm(embed_dim)
        self.ff1 = nn.Linear(embed_dim, ff_dim * 2)
        self.ff2 = nn.Linear(ff_dim, embed_dim)
        self.drop2 = nn.Dropout(dropout)

        if self.use_gated_ffn:
            self.gate = nn.Sequential(
                nn.LayerNorm(embed_dim),
                nn.Linear(embed_dim, embed_dim),
                nn.Sigmoid(),
            )
        else:
            self.gate = None

    def forward(self, x):
        y = self.norm1(x)
        attn_out, _ = self.attn(y, y, y, need_weights=False)
        x = x + self.drop1(attn_out)

        y = self.norm2(x)
        a, b = self.ff1(y).chunk(2, dim=-1)
        y = a * F.gelu(b)
        y = self.ff2(self.drop2(y))

        if self.use_gated_ffn:
            gate = self.gate(x)
            x = x + gate * y
        else:
            x = x + y

        return x


class TabFGT(nn.Module):
    def __init__(
        self,
        input_dim,
        embed_dim=160,
        num_heads=4,
        depth=2,
        ff_mult=2.0,
        dropout=0.10,
        token_dropout=0.01,
        use_feature_identity=True,
        use_gated_ffn=True,
        use_dual_view_readout=True,
    ):
        super().__init__()

        self.use_feature_identity = bool(use_feature_identity)
        self.use_gated_ffn = bool(use_gated_ffn)
        self.use_dual_view_readout = bool(use_dual_view_readout)

        self.tokenizer = NumericFeatureTokenizer(
            n_features=input_dim,
            embed_dim=embed_dim,
            token_dropout=token_dropout,
            use_feature_identity=self.use_feature_identity,
        )

        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.cls_bias = nn.Parameter(torch.zeros(1, 1, embed_dim))

        self.blocks = nn.ModuleList([
            TransformerBlock(
                embed_dim=embed_dim,
                num_heads=num_heads,
                ff_mult=ff_mult,
                dropout=dropout,
                use_gated_ffn=self.use_gated_ffn,
            )
            for _ in range(depth)
        ])

        self.final_norm = nn.LayerNorm(embed_dim)

        head_dim = embed_dim * 2 if self.use_dual_view_readout else embed_dim
        hidden2 = max(embed_dim, 128)

        self.head = nn.Sequential(
            nn.LayerNorm(head_dim),
            nn.Linear(head_dim, hidden2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden2, hidden2 // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden2 // 2, 1),
        )

    def forward(self, x):
        tok = self.tokenizer(x)
        cls = self.cls_token.expand(x.size(0), -1, -1) + self.cls_bias
        tok = torch.cat([cls, tok], dim=1)

        for block in self.blocks:
            tok = block(tok)

        tok = self.final_norm(tok)

        cls_tok = tok[:, 0]

        if self.use_dual_view_readout:
            feat_tok = tok[:, 1:]
            mean_tok = feat_tok.mean(dim=1)
            rep = torch.cat([cls_tok, mean_tok], dim=1)
        else:
            rep = cls_tok

        out = self.head(rep).squeeze(1)
        return out


# ============================================================
# TRAINING AND PREDICTION
# ============================================================

def make_loader(X, y=None, batch_size=1024, shuffle=False):
    X_t = torch.tensor(X, dtype=torch.float32)
    if y is None:
        ds = TensorDataset(X_t)
    else:
        y_t = torch.tensor(y, dtype=torch.float32)
        ds = TensorDataset(X_t, y_t)

    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE.type == "cuda"),
        drop_last=False,
    )


@torch.no_grad()
def predict_proba(model, X, batch_size=1024):
    model.eval()
    loader = make_loader(X, y=None, batch_size=batch_size, shuffle=False)
    probs = []

    for batch in loader:
        xb = batch[0].to(DEVICE, non_blocking=True)
        logits = model(xb)
        p = torch.sigmoid(logits).detach().cpu().numpy()
        probs.append(p)

    return np.concatenate(probs).astype(np.float64)


def train_one_model(
    X_train,
    y_train,
    X_valid,
    y_valid,
    hp,
    ablation_cfg,
    max_epochs=12,
    patience=3,
    seed=42,
):
    set_seed(seed)

    model = TabFGT(
        input_dim=X_train.shape[1],
        embed_dim=hp["embed_dim"],
        num_heads=hp["num_heads"],
        depth=hp["depth"],
        ff_mult=hp["ff_mult"],
        dropout=hp["dropout"],
        token_dropout=hp["token_dropout"],
        use_feature_identity=ablation_cfg["use_feature_identity"],
        use_gated_ffn=ablation_cfg["use_gated_ffn"],
        use_dual_view_readout=ablation_cfg["use_dual_view_readout"],
    ).to(DEVICE)

    train_loader = make_loader(X_train, y_train, batch_size=hp["batch_size"], shuffle=True)
    valid_loader = make_loader(X_valid, y_valid, batch_size=max(1024, hp["batch_size"]), shuffle=False)

    pos_count = max(float(np.sum(np.asarray(y_train) == 1)), 1.0)
    neg_count = max(float(np.sum(np.asarray(y_train) == 0)), 1.0)

    if USE_MILD_POS_WEIGHT:
        pos_weight_value = float(np.clip((neg_count / pos_count) ** POS_WEIGHT_POWER, 1.0, POS_WEIGHT_MAX))
    else:
        pos_weight_value = 1.0

    criterion = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor([pos_weight_value], dtype=torch.float32, device=DEVICE)
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=hp["lr"],
        weight_decay=hp["weight_decay"],
    )

    steps_per_epoch = max(1, len(train_loader))
    total_steps = max(1, max_epochs * steps_per_epoch)

    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=hp["lr"],
        total_steps=total_steps,
        pct_start=0.20,
        div_factor=8.0,
        final_div_factor=80.0,
    )

    use_amp_now = bool(USE_AMP and DEVICE.type == "cuda")
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp_now)

    best_state = copy.deepcopy(model.state_dict())
    best_val_loss = np.inf
    best_epoch = 0
    bad_epochs = 0
    min_delta = 5e-4

    for epoch in range(1, max_epochs + 1):
        model.train()

        for xb, yb in train_loader:
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast(enabled=use_amp_now):
                logits = model(xb)
                loss = criterion(logits, yb)

            scaler.scale(loss).backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

        model.eval()
        val_probs = []
        for xb, _ in valid_loader:
            xb = xb.to(DEVICE, non_blocking=True)
            with torch.cuda.amp.autocast(enabled=use_amp_now):
                logits = model(xb)
                p = torch.sigmoid(logits)
            val_probs.append(p.detach().cpu().numpy())

        val_probs = np.concatenate(val_probs).astype(np.float64)
        val_loss = log_loss(y_valid, clip_probs(val_probs), labels=[0, 1])

        if val_loss < best_val_loss - min_delta:
            best_val_loss = float(val_loss)
            best_epoch = int(epoch)
            best_state = copy.deepcopy(model.state_dict())
            bad_epochs = 0
        else:
            bad_epochs += 1

        if bad_epochs >= patience:
            break

    model.load_state_dict(best_state)
    return model, best_epoch, best_val_loss, pos_weight_value, count_trainable_params(model)


# ============================================================
# DATA LOADING PER VARIANT
# ============================================================

def get_variant_paths(cfg):
    if cfg["use_grafts"]:
        return TRAIN_PATH_GRAFTS, TEST_PATH_GRAFTS
    return TRAIN_PATH_ALL, TEST_PATH_ALL


def load_variant_data(cfg):
    train_path, test_path = get_variant_paths(cfg)
    train_df = safe_read_csv(train_path)
    test_df = safe_read_csv(test_path)

    if TARGET_COLUMN not in train_df.columns:
        raise ValueError(f"Target column '{TARGET_COLUMN}' not found in {train_path}.")

    drop_train = list(set(ID_COLUMNS + DROP_COLUMNS + [TARGET_COLUMN]))
    X_train_df = train_df.drop(columns=[c for c in drop_train if c in train_df.columns], errors="ignore").copy()
    y_raw = train_df[TARGET_COLUMN].copy()

    le = LabelEncoder()
    y = le.fit_transform(y_raw)

    if len(le.classes_) != 2:
        raise ValueError(f"Binary classification only. Found classes: {list(le.classes_)}")

    drop_test = list(set(ID_COLUMNS + DROP_COLUMNS + [TARGET_COLUMN]))
    X_test_df = test_df.drop(columns=[c for c in drop_test if c in test_df.columns], errors="ignore").copy()

    if TARGET_COLUMN in test_df.columns:
        y_test = le.transform(test_df[TARGET_COLUMN])
        has_test_labels = True
    else:
        y_test = None
        has_test_labels = False

    return {
        "train_path": train_path,
        "test_path": test_path,
        "X_train_df": X_train_df,
        "y": y,
        "X_test_df": X_test_df,
        "y_test": y_test,
        "has_test_labels": has_test_labels,
    }


# ============================================================
# ONE TEST-SET ABLATION RUN
# ============================================================

def run_one_ablation_variant(cfg, hp=FIXED_HP):
    data = load_variant_data(cfg)
    X_train_df = data["X_train_df"]
    y = data["y"]
    X_test_df = data["X_test_df"]
    y_test = data["y_test"]
    has_test_labels = data["has_test_labels"]

    print(f"[{now()}] Running {cfg['variant_id']} | {cfg['variant_name']}")

    final_models = []
    final_preps = []
    final_valid_probs = []
    final_valid_targets = []
    final_param_count = None
    final_best_epoch = None
    final_best_val_loss = None

    for seed_i in FINAL_ENSEMBLE_SEEDS:
        X_tr_df, X_va_df, y_tr, y_va = train_test_split(
            X_train_df,
            y,
            test_size=FINAL_VALID_SIZE,
            random_state=seed_i,
            stratify=y,
        )

        prep = FoldPreprocessor()
        X_tr = prep.fit_transform(X_tr_df)
        X_va = prep.transform(X_va_df)

        model, best_epoch, best_val_loss, pos_weight_value, param_count = train_one_model(
            X_tr,
            y_tr,
            X_va,
            y_va,
            hp=hp,
            ablation_cfg=cfg,
            max_epochs=MAX_EPOCHS_ABLATION,
            patience=PATIENCE_ABLATION,
            seed=seed_i,
        )

        final_param_count = param_count
        final_best_epoch = best_epoch
        final_best_val_loss = best_val_loss

        final_models.append(model)
        final_preps.append(prep)

        va_prob = predict_proba(model, X_va, batch_size=max(1024, hp["batch_size"]))
        final_valid_probs.append(va_prob)
        final_valid_targets.append(np.asarray(y_va, dtype=int))

    final_valid_prob = np.concatenate(final_valid_probs).astype(np.float64)
    final_valid_y = np.concatenate(final_valid_targets).astype(int)

    final_threshold = tune_threshold_constrained(
        y_true=final_valid_y,
        prob=final_valid_prob,
        grid=THRESHOLD_GRID,
        acc_drop_tol=ACC_DROP_TOL,
        min_dynamic_precision=MIN_DYNAMIC_PRECISION,
        precision_relax=PRECISION_RELAX,
    )

    test_metrics = None

    test_probs = []
    for model, prep in zip(final_models, final_preps):
        X_test_proc = prep.transform(X_test_df)
        p = predict_proba(model, X_test_proc, batch_size=max(1024, hp["batch_size"]))
        test_probs.append(p)

    final_test_prob = np.mean(np.vstack(test_probs), axis=0)

    if has_test_labels:
        test_metrics = compute_metrics(y_test, final_test_prob, threshold=final_threshold)

    result_row = {
        "variant_id": cfg["variant_id"],
        "variant_name": cfg["variant_name"],
        "grafts": "Yes" if cfg["use_grafts"] else "No",
        "feature_identity": "Yes" if cfg["use_feature_identity"] else "No",
        "gated_ffn": "Yes" if cfg["use_gated_ffn"] else "No",
        "dual_view": "Yes" if cfg["use_dual_view_readout"] else "No",
        "n_features": int(X_train_df.shape[1]),
        "params": int(final_param_count),
        "best_epoch": int(final_best_epoch),
        "best_val_loss": float(final_best_val_loss),
        "test_threshold": float(final_threshold),
    }

    if test_metrics is not None:
        for k, v in test_metrics.items():
            result_row[f"test_{k}"] = v

    return result_row


# ============================================================
# RUN SELECTED TEST-SET ABLATION
# ============================================================

set_seed(SEED)
ensure_dir(OUTPUT_DIR)

print("\n" + "=" * 120)
print(f"[{now()}] SELECTED TEST-SET ABLATION STARTED")
print("=" * 120)
print(f"Device  : {DEVICE}")
print(f"Fixed HP: {hp_to_str(FIXED_HP)}")
print(f"Variants: {len(ABLATION_VARIANTS)}")

all_results = []

for cfg in ABLATION_VARIANTS:
    row = run_one_ablation_variant(cfg, hp=FIXED_HP)
    all_results.append(row)

ablation_results_df = pd.DataFrame(all_results)

if "test_accuracy" in ablation_results_df.columns:
    ablation_results_df = ablation_results_df.sort_values(
        by=["variant_id"],
        ascending=[True]
    ).reset_index(drop=True)

summary_cols = [
    "variant_id",
    "grafts",
    "feature_identity",
    "gated_ffn",
    "dual_view",
    "n_features",
    "test_accuracy",
    "test_precision",
    "test_recall",
    "test_f1",
    "test_roc_auc",
    "test_logloss",
    "test_threshold",
]

print_text_table(
    ablation_results_df.to_dict("records"),
    summary_cols,
    title="SELECTED TEST-SET ABLATION METRICS",
    digits=5,
)

if SAVE_ABLATION_OUTPUTS:
    ablation_results_path = os.path.join(OUTPUT_DIR, "selected_test_ablation_metrics.csv")
    config_path = os.path.join(OUTPUT_DIR, "selected_test_ablation_config.json")

    ablation_results_df.to_csv(ablation_results_path, index=False)

    with open(config_path, "w") as f:
        json.dump(
            {
                "TARGET_COLUMN": TARGET_COLUMN,
                "TRAIN_PATH_ALL": TRAIN_PATH_ALL,
                "TEST_PATH_ALL": TEST_PATH_ALL,
                "TRAIN_PATH_GRAFTS": TRAIN_PATH_GRAFTS,
                "TEST_PATH_GRAFTS": TEST_PATH_GRAFTS,
                "SEED": SEED,
                "FINAL_VALID_SIZE": FINAL_VALID_SIZE,
                "MAX_EPOCHS_ABLATION": MAX_EPOCHS_ABLATION,
                "PATIENCE_ABLATION": PATIENCE_ABLATION,
                "FIXED_HP": FIXED_HP,
                "ABLATION_VARIANTS": ABLATION_VARIANTS,
            },
            f,
            indent=2,
        )

    print("\n" + "=" * 120)
    print("SAVED ABLATION OUTPUTS")
    print("=" * 120)
    print("Summary CSV:", ablation_results_path)
    print("Config JSON:", config_path)

print("\n" + "=" * 120)
print(f"[{now()}] SELECTED TEST-SET ABLATION FINISHED")
print("=" * 120)



[15:15:39] SELECTED TEST-SET ABLATION STARTED
Device  : cuda
Fixed HP: embed=192 | heads=4 | depth=3 | ff_mult=2.0 | drop=0.1 | tok_drop=0.02 | lr=0.0008 | wd=1e-05 | bs=1024
Variants: 8
[15:15:39] Running A01 | All features | No FID | No GATE | No DUAL
[15:15:46] Running A02 | All features | No FID | No GATE | DUAL
[15:15:50] Running A03 | All features | No FID | GATE | DUAL
[15:15:55] Running A04 | GRAFTS | No FID | No GATE | No DUAL
[15:16:00] Running A05 | GRAFTS | No FID | No GATE | DUAL
[15:16:04] Running A06 | GRAFTS | No FID | GATE | No DUAL
[15:16:08] Running A07 | GRAFTS | No FID | GATE | DUAL
[15:16:12] Running A08 | Proposed Full GRAFTS + TabFGT

SELECTED TEST-SET ABLATION METRICS
variant_id | grafts | feature_identity | gated_ffn | dual_view | n_features | test_accuracy | test_precision | test_recall | test_f1 | test_roc_auc | test_logloss | test_threshold
-----------+--------+------------------+-----------+-----------+------------+---------------+----------------+-------